# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.csv

Authors,Title,Publication,Volume,Number,Pages,Year,Publisher
"Thebaud, Thomas; Le Lan, Ga�l; Larcher, Anthony; ",Unsupervised labelling of stolen handwritten digit embeddings with density matching,International Conference on Applied Cryptography and Network Security,,,545-563,2020,Springer International Publishing Cham
"Peillard, Etienne; Thebaud, Thomas; Normand, Jean-Marie; Argelaguet, Ferran; Moreau, Guillaume; L�cuyer, Anatole; ",Virtual objects look farther on the sides: The anisotropy of distance perception in virtual reality,2019 ieee conference on virtual reality and 3d user interfaces (vr),,,227-236,2019,IEEE
"de Vieilleville, Fran�ois; May, St�phane; Lagrange, Adrien; Dupuis, A; Ruiloba, Rosa; Mboula, Fred Ngol�; Bitard-Feildel, Tristan; Nogues, Erwan; Larroche, Corentin; Mazel, Johan; ",Actes de la conf�rence CAID 2020,,,,,2021,
"Thebaud, Thomas; Le Lan, Ga�l; Larcher, Anthony; ",Handwritten digits reconstruction from unlabelled embeddings,"ICASSP 2021-2021 IEEE Internationa

## Import pandas

We are using the very handy pandas library for dataframes.

In [3]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [8]:
publications = pd.read_csv("publications.csv")
publications


,Authors,Title,Publication,Volume,Number,Pages,Year,Publisher
0,"Thebaud, Thomas; Le Lan, Gael; Larcher, Anthony;",Unsupervised labelling of stolen handwritten d...,International Conference on Applied Cryptograp...,NaN,NaN,545-563,2020.0,Springer International Publishing Cham
1,"Peillard, Etienne; Thebaud, Thomas; Normand, J...",Virtual objects look farther on the sides: The...,2019 ieee conference on virtual reality and 3d...,NaN,NaN,227-236,2019.0,IEEE
2,"de Vieilleville, François; May, Stéphane; Lagr...",Actes de la conference CAID 2020,NaN,NaN,NaN,NaN,2021.0,NaN
3,"Thebaud, Thomas; Le Lan, Gael; Larcher, Anthony;",Handwritten digits reconstruction from unlabel...,ICASSP 2021-2021 IEEE International Conference...,NaN,NaN,2540-2544,2021.0,IEEE
4,"Thebaud, Thomas; Le Lan, Gael; Larcher, Anthony;",Spoofing speaker verification with voice style...,2021 IEEE International Workshop on Informatio...,NaN,NaN,7-Jan,2021.0,IEEE
5,"Champion, Pierre; Thebaud, Thomas; Le Lan, Gae...",On the invertibility of a voice privacy system...,2021 IEEE Automatic Speech Recognition and Und...,NaN,NaN,191-197,2021.0,IEEE
6,"Larcher, Anthony;",étiquetage non supervisé de representations de...,Actes de la conference CAID 2020,NaN,NaN,128,NaN,NaN
7,"Thebaud, Thomas;",Attaques par reconstruction de donnees biometr...,NaN,NaN,NaN,NaN,2022.0,Le Mans Université
8,"Kataria, Saurabh; Villalba, Jesus; Moro-Velazq...",Self-FiLM: Conditioning GANs with self-supervi...,arXiv preprint arXiv:2303.03657,NaN,NaN,NaN,2023.0,NaN
9,"Bhati, Saurabhchand; Villalba, Jesus; Moro-Vel...",Segmental speechclip: Utilizing pretrained ima...,Proc. INTERSPEECH,2023.0,NaN,431-435,2023.0,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [9]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md  2015-10-01-paper-title-number-3.md
2010-10-01-paper-title-number-2.md


In [7]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

---
title: "Paper Title Number 1"
collection: publications
permalink: /publication/2009-10-01-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 2009-10-01
venue: 'Journal 1'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).